# 4. Human review: read, rate, and save your own results

**For the two human reviewers and the review coordinator.** This notebook guides the independent
article–topic evaluation. It does not select a model or create topic names.

Each reviewer reads the **same 100 articles** and assesses each model's assigned label separately:
**200 review items per reviewer**. An item is one article paired with one model, so you may see
an article twice. BERTopic's unassigned items receive qualitative flags but no numeric fit rating.

### Your route through the notebook

1. Check with the coordinator that model selection and topic naming are finished.
2. Make a personal working copy, choose your assigned reviewer ID, and load the frozen packet.
3. Read the rubric, then display the next unfinished item.
4. Read the full article, enter your own answers, and explicitly save that item.
5. Repeat steps 3–4; check your progress and resume later whenever needed.
6. When your rows are complete, tell the coordinator. **Do not inspect the other reviewer's scores.**
7. Only after both reviewers finish does the coordinator run the final results section.

Use `Shift+Enter` to run a cell. The configuration and answer-entry cells are the only code
most reviewers need to edit. Start by running cells in order; all save switches are off initially.
After you begin rating, rerun only the display, answer-entry, and progress cells as instructed.

## Before review — coordinator checklist

This is a prerequisite checklist, **not extra work for each reviewer**:

- Notebook 01 reserved the evaluation IDs before development review.
- The team chose the modern configuration and confirmed its labels in notebook 02.
- Laura supplied LDA assignments, terms, topic labels, and a matching corpus manifest.
- Both pipelines' labels are frozen; Laura confirmed the shared review reservation.
- Notebook 03 generated `review_packet.csv`, `review_freeze.json`, `review_status.json`,
  `reviewer_1.csv`, `reviewer_2.csv`, and `rubric.txt` in the selected run's `human_review` folder.
- Assign one person to `reviewer_1` and the other to `reviewer_2`. Keep each person's notebook
  outputs and answer file separate. Provide the common packet plus each person's own CSV.

**Current project boundary:** model selection was deliberately left pending team article review.
If the packet does not exist, finish the checklist first; do not choose a model or invent ratings
just to make this notebook run. Reviewers do not need to rerun embedding or model-fitting cells.

Before entering ratings, use **File → Save Notebook As** to make a personal copy named
`reviewer_1_workbook.ipynb` or `reviewer_2_workbook.ipynb` inside your packet's `human_review`
folder. Those personal workbooks are ignored by Git. Keep this shared notebook as a blank
template; the working copy may contain your answers in both code cells and outputs.

Launch Jupyter from the repository environment with `uv run jupyter lab`. A common local copy
of the frozen packet is required; these full-text files are intentionally not committed to Git.

In [1]:
from pathlib import Path
import html
import json
import sys
import pandas as pd
from IPython.display import HTML, display

# Locate the project whether Jupyter opened from the repository or notebook folder.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ROOT = REPO / "final-project/code/modern-methods"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from src.reviewer import load_session, choose_item, save_answer
from shared.human_review import summarize_reviews

## Step 1 — choose your assigned file

Replace `None` below with **either** `"reviewer_1"` or `"reviewer_2"`, as assigned by the
coordinator. Do not both choose the same ID. Normally leave `PACKET_FOLDER = None`; the
notebook finds the currently selected run. If the coordinator gives you a separate local
folder, enter its absolute path instead.

Loading the session validates the frozen labels and checks your existing answers. It reads
**only your answer file**. Missing project inputs produce a pending message; altered packet
identities or invalid answers raise an error that the coordinator should investigate.

In [2]:
# EDIT THIS: use the reviewer ID assigned to you. Quotes are required.
REVIEWER = None                   # "reviewer_1" or "reviewer_2"

# OPTIONAL: only change this if the coordinator gave you a separate packet folder.
PACKET_FOLDER = None              # Example: Path("/absolute/path/to/human_review")

SESSION = None
CURRENT_ITEM = None
REVIEW_FOLDER = None

if PACKET_FOLDER is not None:
    REVIEW_FOLDER = Path(PACKET_FOLDER).expanduser().resolve()
else:
    selection_file = ROOT / "Results/modern/selection.json"
    selection = json.loads(selection_file.read_text()) if selection_file.exists() else {}
    if selection.get("run_id"):
        REVIEW_FOLDER = ROOT / "Results/comparison" / selection["run_id"] / "human_review"

if REVIEW_FOLDER is None:
    print("Pending: the coordinator must finish model selection, labels, and notebook 03's review-packet step.")
elif REVIEWER is None:
    print("Choose your assigned REVIEWER above, then rerun this cell.")
else:
    required = ["review_packet.csv", "review_freeze.json", "review_status.json", f"{REVIEWER}.csv"]
    missing = [name for name in required if not (REVIEW_FOLDER / name).exists()]
    if missing:
        print("Pending review files:", ", ".join(missing))
        print("Ask the coordinator to supply the frozen packet and your own answer file.")
    else:
        SESSION = load_session(REVIEW_FOLDER, REVIEWER)
        print(f"Opened {REVIEWER}: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
        print("Your answers are saved in:", SESSION["path"])

Pending: the coordinator must finish model selection, labels, and notebook 03's review-packet step.


## Step 2 — use the same rubric throughout

Judge **how well the assigned label describes the article's content**, not whether you agree
with the article or whether its author supports autonomous vehicles. Read the whole article;
do not score from the title or a few matching words alone.

| Fit rating | Meaning | How to decide |
| --- | --- | --- |
| **1** | Unrelated or misleading | The label does not describe the article's subject. |
| **2** | Weak or incidental fit | The label fits a passing mention or a minor part of the article. |
| **3** | Partial or mixed fit | The label describes a meaningful portion, but misses substantial content. |
| **4** | Good fit | The label describes a main theme with minor omissions or imprecision. |
| **5** | Clear central-theme fit | The label accurately and specifically describes a central theme. |

Enter these fields for each item:

| Field | What to enter |
| --- | --- |
| `FIT_RATING` | A whole number 1–5; use `None` for topic −1 (unassigned). |
| `IRRELEVANT` | `True` if the article is not substantively about autonomous/self-driving vehicles; otherwise `False`. An incidental passing mention can be irrelevant. This flag concerns the article, not the model. |
| `AMBIGUOUS_LABEL` | `True` if the wording is unclear, overly broad, or could reasonably mean different things; otherwise `False`. A clearly worded but incorrect label is a poor fit, not automatically ambiguous. For an unassigned item, use `False` because there is no topic label to assess. |
| `NOTES` | Your brief reason or an important limitation. Notes are encouraged for low scores, ambiguity, irrelevance, and unassigned items. Use article IDs and paraphrases rather than copying long passages. |

An article may be irrelevant to autonomous driving yet fit its assigned topic; keep those
judgments separate. Rate each model's label on its own merits, not against the other model.
Do not force the same score for the two models.

**Unassigned example (not a study result):** if the displayed topic is −1, leave
`FIT_RATING = None`, set the two flags, and note what the article discusses. Do not give it a
rating of 1 merely because the model did not assign a topic.

Work independently. Do not use generated scores, model metrics, or another reviewer's ratings
to decide your own. The packets identify the models, so this evaluation is not blinded. If you
recognize an article from earlier development work, record that in your notes and tell the
coordinator; do not silently replace the article or edit the frozen labels.

## Step 3 — display one item and read the full article

Leave `ITEM_NUMBER = None` to open the next unfinished item. To inspect or correct a previous
answer, enter its **1-based item number**. The display includes your existing answers, if any.
Scroll through the article box to read the full text; it is not shortened.

After saving an answer in Step 4, rerun **this cell** to move to the next item.

In [3]:
# Normally leave this as None. For a correction, enter the displayed item number.
ITEM_NUMBER = None
CURRENT_ITEM = None

if SESSION is None:
    print("Finish Step 1 before displaying an article.")
else:
    # Reload your own CSV so an interrupted session resumes from saved answers.
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    CURRENT_ITEM = choose_item(SESSION, ITEM_NUMBER)
    if CURRENT_ITEM is None:
        print("All of your items are complete. Continue to Step 5.")
    else:
        article = SESSION["packet"].iloc[CURRENT_ITEM]
        print(f"ITEM {CURRENT_ITEM + 1} of {len(SESSION['packet'])} — copy this item number into ITEM_TO_SAVE below.")
        display(pd.Series({"article_id": article.article_id, "model": article.model,
                           "topic_id": article.topic_id, "assigned_label": article.label,
                           "title": article.title}, name="Review this assignment"))
        if article.topic_id == -1:
            print("UNASSIGNED: leave FIT_RATING = None; complete the qualitative flags and notes.")
        # Escape article text before rendering: article HTML must be displayed as text.
        display(HTML('<div style="white-space:pre-wrap;max-height:650px;overflow:auto;'
                     'padding:18px;border:1px solid #aaa;line-height:1.6">'
                     + html.escape(str(article.text)) + '</div>'))
        display(SESSION["answers"].loc[CURRENT_ITEM, ["fit_rating", "irrelevant", "ambiguous_label", "notes"]].rename("Your saved answers"))

Finish Step 1 before displaying an article.


## Step 4 — fill in your answer and save it

Edit the values below for the item you just read. Python uses `True`, `False`, and `None`
with capital letters; put written notes inside quotes. Copy the displayed item number into
`ITEM_TO_SAVE`, then set `SAVE_RATING = True` and run the cell.

The item-number check prevents an old answer-entry cell from saving against a newly displayed
article. Only your own row is updated. Saving fails if the packet changed, another session
changed your CSV, a rating is invalid, or you attempt to overwrite an answer without opting in.
For an intentional correction only, set `ALLOW_UPDATE = True` after displaying that item.

**Loop:** display the next item → read → edit every answer field and the item number → save.
Simply running every cell is not a substitute for completing the reviews.

In [4]:
# FILL THESE IN for the displayed item; no answers are supplied automatically.
ITEM_TO_SAVE = None          # Copy the displayed ITEM number here (e.g., 1).
FIT_RATING = None            # 1, 2, 3, 4, or 5. Keep None only for an unassigned item.
IRRELEVANT = None            # Replace with True or False.
AMBIGUOUS_LABEL = None       # Replace with True or False.
NOTES = ""                  # Write your reason or any uncertainty in your own words.

SAVE_RATING = False          # Change to True when this item's answers are ready to save.
ALLOW_UPDATE = False         # True only when intentionally correcting existing answers.

if not SAVE_RATING:
    print("Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.")
elif SESSION is None or CURRENT_ITEM is None:
    print("Nothing saved. Display an item in Step 3 first.")
elif type(ITEM_TO_SAVE) is not int or ITEM_TO_SAVE != CURRENT_ITEM + 1:
    raise ValueError("ITEM_TO_SAVE must match the displayed item's number. Check it before saving.")
else:
    SESSION = save_answer(SESSION, CURRENT_ITEM, FIT_RATING, IRRELEVANT, AMBIGUOUS_LABEL,
                          notes=NOTES, allow_update=ALLOW_UPDATE)
    print(f"Saved item {ITEM_TO_SAVE} to {SESSION['reviewer']}.csv.")
    print(f"Progress: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
    print("Rerun Step 3 to display the next item, then update all answer fields.")
    CURRENT_ITEM = None      # A new display is required before another save.
SAVE_RATING = False

Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.


## Step 5 — check progress, take a break, or finish

This cell validates **your own** file and lists unfinished item numbers. Blank flags remain
unfinished, even for unassigned articles. Notes are optional, but useful explanations should
accompany difficult judgments.

To take a break, save your current item, then close Jupyter. To resume, reopen the notebook,
run Steps 1–3 with the same reviewer ID, and leave `ITEM_NUMBER = None`.

When complete, give the coordinator your `reviewer_1.csv` or `reviewer_2.csv` through the team's
usual file-sharing process. Keep the original independent ratings; discuss disagreements only
after both reviewers finish. Do not overwrite them with consensus ratings.

Your personal workbook may contain full article text in outputs and ratings in edited code.
Share your CSV with the coordinator, not your working notebook. If preparing a blank notebook
template to share, reset the answer/configuration cells and save switches **as well as** using
**Kernel → Restart Kernel and Clear Outputs** (wording varies by Jupyter version). Clearing
outputs alone does not erase ratings typed into code cells.

In [5]:
if SESSION is None:
    print("Load your assigned review file in Step 1 first.")
else:
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    total = len(SESSION["packet"])
    finished = int(SESSION["complete"].sum())
    print(f"{SESSION['reviewer']}: {finished}/{total} items complete ({finished / total:.0%}).")
    remaining = SESSION["answers"].loc[~SESSION["complete"], ["article_id", "model", "topic_id"]].copy()
    remaining.insert(0, "item_number", remaining.index + 1)
    if remaining.empty:
        print("Your file is complete and valid. Tell the coordinator; keep your independent ratings unchanged.")
    else:
        print("Next unfinished items (first 20):")
        display(remaining.head(20))
    print("Your saved file:", SESSION["path"])

Load your assigned review file in Step 1 first.


## Step 6 — coordinator only: summarize both completed files

**Reviewers stop here until both have finished independently.** This section reads both answer
files and reports agreement. The coordinator should collect the two CSVs into the frozen packet
folder without changing IDs, topics, or labels.

Set both switches below to `True` only after both reviewers have submitted their independent
ratings. The summary validates the complete files; if anything is missing it reports pending.
Notebook 03 can generate the same summary later for the report.

| Output | Meaning |
| --- | --- |
| `sample_articles` / `rated_articles` | Articles sampled and articles with numeric fit ratings for each model. |
| `unassigned_articles` | BERTopic sample items excluded from numeric ratings because topic = −1. |
| `mean_1`, `mean_2` | Each reviewer's mean assigned-article fit score. |
| `exact_agreement` | Fraction of paired ratings that are identical. |
| `within_one_agreement` | Fraction differing by at most one point. |
| `weighted_kappa` | Quadratic-weighted agreement beyond chance; larger differences count more. |
| `kappa_ci_low`, `kappa_ci_high` | 95% paired-article bootstrap interval, using 1,000 resamples. |
| `valid_bootstrap_samples` | Resamples with defined kappa; constant ratings can make kappa undefined. |

Agreement measures consistency between reviewers, not model accuracy. Keep the unassigned
coverage and conditional rating denominators visible when comparing models. The same articles
were reserved from development review but were still used in unsupervised model fitting.
The separate supplemental `outlier_review.csv` is qualitative and is not added to the 100-article
agreement sample.

In [6]:
# COORDINATOR ONLY: leave both False during independent review.
BOTH_REVIEWERS_FINISHED = False
RUN_TEAM_SUMMARY = False
TEAM_SUMMARY = None

if BOTH_REVIEWERS_FINISHED and RUN_TEAM_SUMMARY:
    if REVIEW_FOLDER is None:
        print("Choose the frozen packet folder in Step 1 first.")
    else:
        TEAM_SUMMARY = summarize_reviews(REVIEW_FOLDER, bootstrap_samples=1000, seed=42)
        display(TEAM_SUMMARY)
        if isinstance(TEAM_SUMMARY, pd.DataFrame):
            print("Saved human_evaluation.csv and rating_distributions.csv in", REVIEW_FOLDER)
            print("Rerun notebook 03 to refresh the PR3 evidence checklist.")
else:
    print("Team summary not run; independent reviewer scores have not been compared.")

Team summary not run; independent reviewer scores have not been compared.


## Step 7 — coordinator/team: fill in the interpretation

Complete the notes below **after** the computed summary is available. Explain what the evidence
supports; do not fill blanks with assumed findings. Suggested prompts:

- How well do the labels describe assigned articles? Compare both reviewer means and distributions.
- Where did reviewers disagree, and were labels broad, unclear, or articles multi-topic?
- Which unassigned or irrelevant cases limit interpretation? Use IDs and short paraphrases.
- How do conditional rating coverage, the small sample, two team reviewers, and unblinded model
  identities limit generalization? Do not describe an undefined kappa as zero agreement.

The optional save cell exports a short `team_interpretation.csv` alongside the numerical summary.
These are human-written notes for the report; this notebook never edits the PR3 manuscript.

In [7]:
# Replace the empty strings with your team's interpretation of the completed results.
TEAM_NOTES = {
    "article_topic_fit": "",
    "reviewer_disagreements": "",
    "unassigned_or_irrelevant_examples": "",
    "limitations_and_report_wording": "",
}
SAVE_TEAM_NOTES = False

if SAVE_TEAM_NOTES:
    if not isinstance(TEAM_SUMMARY, pd.DataFrame):
        raise ValueError("Run a complete team summary before saving interpretation notes.")
    if any(not text.strip() for text in TEAM_NOTES.values()):
        raise ValueError("Fill in each interpretation field, or state that no conclusion is supported.")
    notes_table = pd.DataFrame(TEAM_NOTES.items(), columns=["section", "human_interpretation"])
    notes_table.to_csv(REVIEW_FOLDER / "team_interpretation.csv", index=False)
    display(notes_table)
else:
    print("No interpretation notes saved. Fill these in after both reviewers finish.")

No interpretation notes saved. Fill these in after both reviewers finish.
